# Constrained MOPSO Pipeline (Scalar-Sum Variant)

**Paper:** *Constrained Multi-Objective Particle Swarm Optimization for Sensitivity-Aware Pruning of Lung Cancer Histopathology Networks*

## 0. Setup

In [ ]:
# Uncomment if any dependency is missing:
# !pip install torch torchvision numpy pandas scikit-learn matplotlib tqdm pillow

import sys
print(f'Python {sys.version.split()[0]}')

In [ ]:
import copy, json, math, os, random, time
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils.prune as prune
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score)
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, models, transforms
from tqdm.auto import tqdm

print(f'PyTorch     : {torch.__version__}')
print(f'CUDA avail  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA version: {torch.version.cuda}')
    print(f'GPU         : {torch.cuda.get_device_name(0)}')

## 1. Configuration


In [ ]:
SEED = 42                    

DATASET = 'LungHist700'       # 'LungHist700' or 'LC25000'
DATA_DIR = {
    'LungHist700': 'data/LungHist700',
    'LC25000':     'data/LC25000',
}[DATASET]
NUM_CLASSES = 3

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

IMG_SIZE          = 224
BATCH_SIZE        = 32
NUM_WORKERS       = 0

EPOCHS_PRETRAIN   = 15        
EPOCHS_FINETUNE   = 10       
LR_PRETRAIN       = 1e-4     
LR_FINETUNE       = 1e-5    
WEIGHT_DECAY      = 1e-4     

PROBING_RATES   = [0.3, 0.5, 0.7]
DROP_TOLERANCE  = 0.05
NCAL_CAP        = 500

N_PARTICLES      = 30         
N_ITERATIONS     = 50        
ARCHIVE_SIZE     = 50
INERTIA_W        = 0.5
COGNITIVE_C1     = 1.5
SOCIAL_C2        = 1.5

PENALTY_LAMBDA   = 10.0
SENS_THRESHOLD_TAU = 0.90

NOMINAL_SMAX    = 0.95

DETERMINISTIC_CUDNN = True

RESULTS_DIR = Path(f'results/{DATASET}_seed{SEED}_scalar')
CKPT_DIR    = Path(f'checkpoints/{DATASET}_seed{SEED}_scalar')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Dataset  : {DATASET}')
print(f'Seed     : {SEED}')
print(f'Variant  : scalar-sum')
print(f'Results  : {RESULTS_DIR}')
print(f'Checkpoints: {CKPT_DIR}')

### 1.1 Reproducibility seeding

In [ ]:
def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if DETERMINISTIC_CUDNN:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True

set_global_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device       : {DEVICE}')
print(f'cuDNN det.   : {torch.backends.cudnn.deterministic}')

### 1.2 Results container

In [ ]:
RESULTS_FILE = RESULTS_DIR / 'all_results.json'
results = {}

def save_results():
    with open(RESULTS_FILE, 'w') as f:
        json.dump(results, f, indent=2, default=str)

def log_result(exp_id, data):
    results[exp_id] = data
    save_results()
    print(f'[{exp_id}] saved to {RESULTS_FILE}')

## 2. Data loading and splits

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

assert os.path.isdir(DATA_DIR), f'Dataset folder not found: {DATA_DIR}'

LUNGHIST_SUPERCLASS = {
    'aca_bd': 'aca', 'aca_md': 'aca', 'aca_pd': 'aca',
    'scc_bd': 'scc', 'scc_md': 'scc', 'scc_pd': 'scc',
    'nor':    'nor',
}

class ImageFolderWithMapping(datasets.ImageFolder):
    def __init__(self, root, transform=None, class_mapping=None):
        super().__init__(root, transform=transform)
        if class_mapping is None:
            self.super_classes = self.classes
            return
        super_class_names = sorted(set(class_mapping.values()))
        name_to_idx = {n: i for i, n in enumerate(super_class_names)}
        self.super_classes = super_class_names
        self.samples = [
            (p, name_to_idx[class_mapping[self.classes[old]]])
            for p, old in self.samples
        ]
        self.targets = [s[1] for s in self.samples]

def build_image_folder(root, transform):
    if DATASET == 'LungHist700':
        return ImageFolderWithMapping(root, transform=transform,
                                       class_mapping=LUNGHIST_SUPERCLASS)
    return datasets.ImageFolder(root, transform=transform)

full_train = build_image_folder(DATA_DIR, transform=train_tf)
full_eval  = build_image_folder(DATA_DIR, transform=eval_tf)
CLASS_NAMES = (full_train.super_classes
               if hasattr(full_train, 'super_classes')
               else full_train.classes)
print(f'Dataset size: {len(full_train)}')
print(f'Classes     : {CLASS_NAMES}')

In [ ]:
n_total = len(full_train)
n_train = int(TRAIN_RATIO * n_total)
n_val   = int(VAL_RATIO   * n_total)
n_test  = n_total - n_train - n_val

split_generator = torch.Generator().manual_seed(SEED)
train_idx, val_idx, test_idx = random_split(
    range(n_total), [n_train, n_val, n_test], generator=split_generator
)
train_idx, val_idx, test_idx = list(train_idx), list(val_idx), list(test_idx)

train_set = Subset(full_train, train_idx)
val_set   = Subset(full_eval,  val_idx)
test_set  = Subset(full_eval,  test_idx)

train_targets = np.array([full_train.targets[i] for i in train_idx])
class_counts = np.bincount(train_targets, minlength=NUM_CLASSES)
CLASS_WEIGHTS = torch.tensor(
    class_counts.sum() / (NUM_CLASSES * class_counts), dtype=torch.float32
).to(DEVICE)
print(f'Train: {len(train_set)}, Val: {len(val_set)}, Test: {len(test_set)}')
print(f'Class counts (train): {class_counts.tolist()}')

In [ ]:
def make_loader(ds, shuffle=False):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True)

train_loader = make_loader(train_set, shuffle=True)
val_loader   = make_loader(val_set)
test_loader  = make_loader(test_set)

## 3. Backbone network

In [ ]:
def build_resnet50(num_classes=NUM_CLASSES, pretrained=True):
    weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
    model = models.resnet50(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(DEVICE)

def get_prunable_layers(model):
    layers = []
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            if isinstance(module, nn.Linear) and module.out_features == NUM_CLASSES:
                continue
            layers.append((name, module))
    return layers

_tmp = build_resnet50(pretrained=False)
N_PRUNABLE = len(get_prunable_layers(_tmp))
print(f'Number of prunable layers: {N_PRUNABLE}')
assert N_PRUNABLE == 53
del _tmp

## 4. Helper functions


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()
        out = model(x); loss = criterion(out, y); loss.backward(); optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += x.size(0)
    return total_loss / total, correct / total

def compute_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    per_class_recall = cm.diagonal() / cm.sum(axis=1).clip(min=1)
    out = {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'sens_min': float(per_class_recall.min()),
    }
    for i, c in enumerate(CLASS_NAMES):
        out[f'sens_{c}'] = float(per_class_recall[i])
    return out

@torch.no_grad()
def eval_model(model, loader):
    model.eval()
    all_y, all_pred = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        all_y.extend(y.tolist())
        all_pred.extend(model(x).argmax(1).cpu().tolist())
    return compute_metrics(np.array(all_y), np.array(all_pred))

def apply_sparsity_vector(model, sparsity_vec):
    layers = get_prunable_layers(model)
    assert len(layers) == len(sparsity_vec)
    for (name, module), s in zip(layers, sparsity_vec):
        s = float(max(0.0, min(NOMINAL_SMAX, s)))
        if s > 0:
            prune.l1_unstructured(module, name='weight', amount=s)

def fuse_pruning_masks(model):
    for name, module in get_prunable_layers(model):
        try: prune.remove(module, 'weight')
        except ValueError: pass

def actual_sparsity(model):
    total, zero = 0, 0
    for name, module in get_prunable_layers(model):
        total += module.weight.numel()
        zero  += (module.weight == 0).sum().item()
    return zero / max(1, total)

def get_layer_info(model):
    params, flops = [], []
    handles, output_shapes = [], {}
    def hook_fn(name):
        def hook(mod, inp, out): output_shapes[name] = tuple(out.shape)
        return hook
    for name, module in get_prunable_layers(model):
        handles.append(module.register_forward_hook(hook_fn(name)))
    model.eval()
    with torch.no_grad():
        _ = model(torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE))
    for h in handles: h.remove()
    for name, module in get_prunable_layers(model):
        params.append(module.weight.numel())
        out_shape = output_shapes[name]
        if isinstance(module, nn.Conv2d):
            c_in = module.weight.shape[1]
            k_h, k_w = module.weight.shape[2], module.weight.shape[3]
            c_out = module.out_channels
            _, _, h, w = out_shape
            fl = 2 * (c_in // module.groups) * k_h * k_w * c_out * h * w
        else:
            fl = 2 * module.in_features * module.out_features
        flops.append(int(fl))
    return params, flops

def flops_params_ratio(sparsity_vec, lp, lf):
    s = np.asarray(sparsity_vec); p = np.asarray(lp); f = np.asarray(lf)
    return float((f * (1 - s)).sum() / f.sum()), float((p * (1 - s)).sum() / p.sum())

## 5. Experiment 1: Dense baseline training

In [ ]:
dense_path = CKPT_DIR / 'dense_resnet50.pt'
model = build_resnet50(pretrained=True)
criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)

if dense_path.exists():
    print(f'Loading existing checkpoint from {dense_path}')
    model.load_state_dict(torch.load(dense_path, map_location=DEVICE))
else:
    optimizer = torch.optim.AdamW(model.parameters(),
                                   lr=LR_PRETRAIN, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_PRETRAIN)
    best_val_acc = 0.0
    for epoch in range(EPOCHS_PRETRAIN):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_metrics = eval_model(model, val_loader)
        scheduler.step()
        print(f'  Epoch {epoch + 1:2d}/{EPOCHS_PRETRAIN} | '
              f'train acc {tr_acc:.4f} | val acc {val_metrics["accuracy"]:.4f}')
        if val_metrics['accuracy'] > best_val_acc:
            best_val_acc = val_metrics['accuracy']
            torch.save(model.state_dict(), dense_path)
    model.load_state_dict(torch.load(dense_path, map_location=DEVICE))

dense_metrics = eval_model(model, test_loader)
_, dense_flops = get_layer_info(model)
dense_params = sum(p.numel() for p in model.parameters())
print(f'\nDense test metrics: {dense_metrics}')

log_result('E1', {
    'metrics': dense_metrics, 'num_layers': N_PRUNABLE,
    'total_params': dense_params,
    'total_flops_per_image': int(sum(dense_flops)),
})

## 6. Experiment 2: Per-layer sensitivity analysis


In [ ]:
NCAL = min(NCAL_CAP, len(val_set))
cal_indices = np.random.RandomState(SEED).choice(len(val_set), size=NCAL,
                                                  replace=False).tolist()
cal_subset = Subset(val_set, cal_indices)
cal_loader = make_loader(cal_subset)
dense_cal_metrics = eval_model(model, cal_loader)
acc_dense = dense_cal_metrics['accuracy']
print(f'Calibration subset: N_cal = {NCAL}')
print(f'Dense accuracy on calibration subset: {acc_dense:.4f}')

In [ ]:
def apply_eq8(drops, tolerance=DROP_TOLERANCE):
    d_low, d_mid, _ = drops
    if d_low > tolerance:
        return 0.30
    elif d_mid > tolerance:
        return 0.50
    else:
        return 0.80

print('=' * 60); print('E2 : Per-layer zero-shot sensitivity analysis'); print('=' * 60)
sensitivity_log = {}
for layer_idx in tqdm(range(N_PRUNABLE), desc='Layers'):
    drops_at_rates = []
    for s in PROBING_RATES:
        trial = copy.deepcopy(model)
        sv = [0.0] * N_PRUNABLE
        sv[layer_idx] = s
        apply_sparsity_vector(trial, sv)
        acc_pruned = eval_model(trial, cal_loader)['accuracy']
        drops_at_rates.append(acc_dense - acc_pruned)
        fuse_pruning_masks(trial)
        del trial
    sensitivity_log[str(layer_idx)] = drops_at_rates

SMAX_PER_LAYER = [apply_eq8(sensitivity_log[str(i)]) for i in range(N_PRUNABLE)]
SENSITIVE_LAYERS = [i for i in range(N_PRUNABLE) if SMAX_PER_LAYER[i] == 0.30]

print(f'\nSensitivity analysis complete.')
print(f'  s_max = 0.30: {len(SENSITIVE_LAYERS)}')
print(f'  s_max = 0.50: {sum(1 for v in SMAX_PER_LAYER if v == 0.50)}')
print(f'  s_max = 0.80: {sum(1 for v in SMAX_PER_LAYER if v == 0.80)}')

log_result('E2', {
    'sensitivity_per_layer': sensitivity_log,
    's_max_per_layer': SMAX_PER_LAYER,
    'sensitive_layers': SENSITIVE_LAYERS,
    'n_sensitive': len(SENSITIVE_LAYERS),
})

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for j, s in enumerate(PROBING_RATES):
    drops = [sensitivity_log[str(i)][j] for i in range(N_PRUNABLE)]
    ax.plot(range(N_PRUNABLE), drops, marker='o', markersize=3, label=f's={s}')
ax.axhline(DROP_TOLERANCE, color='red', linestyle='--', alpha=0.6,
           label=f'tolerance {int(DROP_TOLERANCE * 100)}%')
ax.set_xlabel('Layer index'); ax.set_ylabel('Accuracy drop')
ax.set_title(f'Per-layer sensitivity ({DATASET}, seed {SEED}, scalar variant)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig4_sensitivity.png', dpi=120)
plt.show()

## 7. Zero-shot proxy fitness

In [ ]:
LAYER_PARAMS, LAYER_FLOPS = get_layer_info(model)

def proxy_fitness(sparsity_vec):
    trial = copy.deepcopy(model)
    apply_sparsity_vector(trial, sparsity_vec)
    m = eval_model(trial, cal_loader)
    fuse_pruning_masks(trial)
    fr, pr = flops_params_ratio(sparsity_vec, LAYER_PARAMS, LAYER_FLOPS)
    out = {'accuracy': m['accuracy'], 'sens_min': m['sens_min'],
           'flops_ratio': fr, 'params_ratio': pr}
    del trial
    return out

_sanity = proxy_fitness([0.0] * N_PRUNABLE)
print(f'Proxy sanity check (all zeros):')
print(f'  accuracy: {_sanity["accuracy"]:.4f} (should match dense cal: {acc_dense:.4f})')
print(f'  flops   : {_sanity["flops_ratio"]:.4f} (should be 1.0)')
print(f'  params  : {_sanity["params_ratio"]:.4f} (should be 1.0)')

## 8. Experiment 7: Constrained MOPSO (scalar-sum variant)

This section implements the scalar-sum variant of the search:

- **Personal best update**: a particle's personal best is replaced when the sum of its three objectives decreases. This is a scalarization of the Pareto comparison used in the canonical variant and biases personal-best selection toward solutions with low combined objective score rather than solutions non-dominated with respect to the current personal best.

- **Archive truncation**: when the archive exceeds `A` members, it is sorted by the sum of objectives and the first `A` are retained. This differs from NSGA-II crowding distance, which prioritizes spread. The practical effect is that the archive concentrates on solutions with low combined score, potentially at the cost of representation along the Pareto front.

- **Leader selection**: uniform random from the current archive (identical to the canonical variant).

- **Position clipping**: every particle is clipped to `[0, s_max]` after each update (identical to the canonical variant).

The archive, pbest, and velocity dynamics otherwise follow the canonical PSO update rule.

In [ ]:
def dominates(a, b):
    """Used only for archive insertion (removing dominated existing members)."""
    a, b = np.asarray(a), np.asarray(b)
    return bool(np.all(a <= b) and np.any(a < b))


class ScalarSumMOPSO:
    """Constrained MOPSO (scalar-sum variant).

    Objectives (minimized jointly, with Pareto-based archive maintenance but
    scalar-sum personal-best updates and archive truncation):
        F1' = (1 - accuracy) + lambda * [max(0, tau - sens_min)]^2
        F2  = FLOPs ratio
        F3  = parameters ratio
    """

    def __init__(self, n_particles, n_dims, smax_vec,
                 w=INERTIA_W, c1=COGNITIVE_C1, c2=SOCIAL_C2,
                 archive_size=ARCHIVE_SIZE,
                 penalty_lambda=PENALTY_LAMBDA,
                 sens_threshold=SENS_THRESHOLD_TAU, seed=SEED):
        self.n_particles = n_particles
        self.n_dims = n_dims
        self.smax = np.array(smax_vec, dtype=float)
        self.w, self.c1, self.c2 = w, c1, c2
        self.archive_size = archive_size
        self.lam = penalty_lambda
        self.tau = sens_threshold

        self.rng_init = np.random.RandomState(seed)
        self.rng_loop = np.random.RandomState(seed + 1)

        self.X = self.rng_init.uniform(0, 1, (n_particles, n_dims)) * self.smax[None, :]
        self.V = self.rng_init.uniform(-0.1, 0.1, (n_particles, n_dims))
        self.pbest_X = self.X.copy()
        self.pbest_F = [None] * n_particles
        self.archive = []

    def evaluate(self, x):
        x = np.clip(x, 0.0, self.smax)
        p = proxy_fitness(x.tolist())
        penalty = self.lam * max(0.0, self.tau - p['sens_min']) ** 2
        F = (1.0 - p['accuracy'] + penalty, p['flops_ratio'], p['params_ratio'])
        return F, p, x

    def update_pbest(self, i, F_new, x_new):
        """SCALAR-SUM VARIANT: replace pbest when sum(F_new) decreases."""
        if self.pbest_F[i] is None or sum(F_new) < sum(self.pbest_F[i]):
            self.pbest_X[i] = x_new.copy()
            self.pbest_F[i] = F_new

    def update_archive(self, x_new, F_new, p_new):
        """Pareto-based admission; scalar-sum truncation when archive overflows."""
        for _, aF, _ in self.archive:
            if dominates(aF, F_new):
                return
        self.archive = [(ax, aF, ap) for ax, aF, ap in self.archive
                        if not dominates(F_new, aF)]
        self.archive.append((x_new.copy(), F_new, p_new))
        if len(self.archive) > self.archive_size:
            # SCALAR-SUM VARIANT: sort by sum(F) and keep first A
            sorted_arch = sorted(self.archive, key=lambda t: sum(t[1]))
            self.archive = sorted_arch[:self.archive_size]

    def pick_leader(self):
        if not self.archive:
            return self.X[self.rng_loop.randint(self.n_particles)]
        return self.archive[self.rng_loop.randint(len(self.archive))][0]

    def run(self, n_iter):
        for it in tqdm(range(n_iter), desc='MOPSO (scalar)'):
            for i in range(self.n_particles):
                F, p, x_clip = self.evaluate(self.X[i])
                self.X[i] = x_clip
                self.update_pbest(i, F, x_clip)
                self.update_archive(x_clip, F, p)
            for i in range(self.n_particles):
                r1 = self.rng_loop.uniform(0, 1, self.n_dims)
                r2 = self.rng_loop.uniform(0, 1, self.n_dims)
                leader = self.pick_leader()
                self.V[i] = (
                    self.w * self.V[i]
                    + self.c1 * r1 * (self.pbest_X[i] - self.X[i])
                    + self.c2 * r2 * (leader - self.X[i])
                )
                self.V[i] = np.clip(self.V[i], -0.2, 0.2)
                self.X[i] = np.clip(self.X[i] + self.V[i], 0.0, self.smax)
        return self.archive

In [ ]:

t_start = time.time()
mopso = ScalarSumMOPSO(
    n_particles=N_PARTICLES,
    n_dims=N_PRUNABLE,
    smax_vec=SMAX_PER_LAYER,
)
pareto = mopso.run(n_iter=N_ITERATIONS)
mopso_wall_time = time.time() - t_start

print(f'\nMOPSO complete.')
print(f'  Archive size: {len(pareto)}')
print(f'  Wall time   : {mopso_wall_time:.1f} s')

archive_data = []
for x, F, p in pareto:
    archive_data.append({
        'sparsity_vec': x.tolist(),
        'proxy_acc':    p['accuracy'],
        'flops_ratio':  p['flops_ratio'],
        'params_ratio': p['params_ratio'],
        'sens_min':     p['sens_min'],
        'penalized_f1': F[0],
        'feasible':     p['sens_min'] >= SENS_THRESHOLD_TAU,
    })

log_result('E7', {
    'archive': archive_data,
    'archive_size': len(pareto),
    'wall_time_s': mopso_wall_time,
    'n_particles': N_PARTICLES,
    'n_iterations': N_ITERATIONS,
    'variant': 'scalar_sum',
})

In [ ]:
flops_arr = np.array([t[1][1] for t in pareto])
acc_arr   = np.array([t[2]['accuracy'] for t in pareto])
sens_arr  = np.array([t[2]['sens_min'] for t in pareto])
feasible  = sens_arr >= SENS_THRESHOLD_TAU

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(flops_arr[feasible], acc_arr[feasible],
           s=50, c='C0', marker='o', edgecolors='white',
           label=f'Feasible (n={feasible.sum()})')
ax.scatter(flops_arr[~feasible], acc_arr[~feasible],
           s=50, c='C3', marker='x',
           label=f'Constraint-violating (n={(~feasible).sum()})')
ax.set_xlabel('FLOPs ratio'); ax.set_ylabel('Proxy accuracy')
ax.set_title(f'MOPSO Pareto archive ({DATASET}, seed {SEED}, scalar variant)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig5_pareto.png', dpi=120)
plt.show()

## 9. Operating point selection and fine-tuning


In [ ]:
def select_operating_points(archive, prefer_feasible=True):
    pool = archive
    feasible_pool = [t for t in archive if t[2]['sens_min'] >= SENS_THRESHOLD_TAU]
    if prefer_feasible and feasible_pool:
        pool = feasible_pool
        print(f'Selecting from {len(pool)} feasible archive members.')
    else:
        print(f'WARNING: no feasible members, selecting from full archive (n={len(pool)}).')
    F_mat = np.array([t[1] for t in pool], dtype=float)
    f1, f2, f3 = F_mat[:, 0], F_mat[:, 1], F_mat[:, 2]
    high_acc_idx  = int(np.argmin(f1))
    high_comp_idx = int(np.argmin((f2 + f3) / 2.0))
    f_min = F_mat.min(axis=0); f_max = F_mat.max(axis=0)
    range_ = np.where(f_max - f_min > 0, f_max - f_min, 1.0)
    normed = (F_mat - f_min) / range_
    dists = np.linalg.norm(normed, axis=1)
    balanced_idx = int(np.argmin(dists))
    return {
        'high_acc':  pool[high_acc_idx],
        'balanced':  pool[balanced_idx],
        'high_comp': pool[high_comp_idx],
    }

selected = select_operating_points(pareto)
for key, (x, F, p) in selected.items():
    print(f'  {key:10s}: proxy_acc={p["accuracy"]:.4f} | '
          f'flops={p["flops_ratio"]:.3f} | params={p["params_ratio"]:.3f} | '
          f'sens_min={p["sens_min"]:.4f}')

In [ ]:
def finetune_pruned_model(sparsity_vec, epochs=EPOCHS_FINETUNE, verbose=False):
    pruned = build_resnet50(pretrained=False)
    pruned.load_state_dict(torch.load(dense_path, map_location=DEVICE))
    apply_sparsity_vector(pruned, sparsity_vec)
    optimizer = torch.optim.AdamW(pruned.parameters(),
                                   lr=LR_FINETUNE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    best_val_acc, best_state = 0.0, None
    for epoch in range(epochs):
        tr_loss, tr_acc = train_one_epoch(pruned, train_loader, optimizer, criterion)
        val_metrics = eval_model(pruned, val_loader); scheduler.step()
        if verbose:
            print(f'    Epoch {epoch + 1:2d}/{epochs} | '
                  f'train acc {tr_acc:.4f} | val acc {val_metrics["accuracy"]:.4f}')
        if val_metrics['accuracy'] > best_val_acc:
            best_val_acc = val_metrics['accuracy']
            best_state = copy.deepcopy(pruned.state_dict())
    pruned.load_state_dict(best_state)
    fuse_pruning_masks(pruned)
    return pruned

print('=' * 60); print('E8 : Fine-tune 3 selected operating points'); print('=' * 60)
finetuned = {}
for key, (x, F, p) in selected.items():
    print(f'\n--- {key} ---')
    sv = x.tolist()
    ft_model = finetune_pruned_model(sv, verbose=True)
    ft_metrics = eval_model(ft_model, test_loader)
    fr, pr = flops_params_ratio(sv, LAYER_PARAMS, LAYER_FLOPS)
    as_ = actual_sparsity(ft_model)
    finetuned[key] = {
        'sparsity_vec':    sv,
        'test_metrics':    ft_metrics,
        'flops_ratio':     fr,
        'params_ratio':    pr,
        'actual_sparsity': as_,
        'proxy_accuracy':  p['accuracy'],
        'proxy_sens_min':  p['sens_min'],
    }
    torch.save(ft_model.state_dict(), CKPT_DIR / f'ft_{key}.pt')
    print(f'  Test metrics: {ft_metrics}')
    print(f'  FLOPs ratio: {fr:.4f}, Params ratio: {pr:.4f}')
    del ft_model

log_result('E8', {
    'finetuned_solutions': finetuned,
    'selected_from_archive_size': len(pareto),
})

## 10. Summary

In [ ]:
print('=' * 60)
print(f'Scalar-sum variant pipeline complete for {DATASET}, seed {SEED}')
print('=' * 60)
print(f'\nResults written to : {RESULTS_FILE}')
print(f'\nSummary of fine-tuned operating points:')
print(f'{"Point":<12} {"Accuracy":>10} {"Sens min":>10} {"FLOPs":>8} {"Params":>8}')
for key, data in finetuned.items():
    m = data['test_metrics']
    print(f'{key:<12} {m["accuracy"]:>10.4f} {m["sens_min"]:>10.4f} '
          f'{data["flops_ratio"]:>8.3f} {data["params_ratio"]:>8.3f}')